In [11]:
import sys, time, hashlib, requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "books_raw.csv"

sys.path.append(str(ROOT / "src"))
from config import SEED
print("SEED =", SEED)

SEED = 3020


### What above cell does:
This cell imports the libraries and sets the file paths. It also loads the SEED from the config file. The SEED is 3020, which is the last four digits of the lowest roll number in our team. The output shows SEED = 3020, so the config file loaded correctly.

In [2]:
START_URL = "https://books.toscrape.com/catalogue/page-1.html"
SLEEP = 0.5
MAX_RETRIES = 3

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (university project scraper)"})
request_count = 0
failed_urls = []

def get_soup(url):
    global request_count
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request_count += 1
            r = session.get(url, timeout=15)
            r.raise_for_status()
            r.encoding = "utf-8"
            time.sleep(SLEEP)
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"[retry {attempt}/{MAX_RETRIES}] {url} -> {e}")
            time.sleep(2 * attempt)
    failed_urls.append(url)
    return None

def get_product_urls():
    urls, pages, page_url = [], 0, START_URL
    while page_url:
        soup = get_soup(page_url)
        if soup is None:
            break
        pages += 1
        urls += [urljoin(page_url, a["href"]) for a in soup.select("article.product_pod h3 a")]
        nxt = soup.select_one("li.next a")
        page_url = urljoin(page_url, nxt["href"]) if nxt else None
    return urls, pages

def parse_product(url):
    soup = get_soup(url)
    if soup is None:
        return None
    table = {tr.th.get_text(strip=True): tr.td.get_text(strip=True)
             for tr in soup.select("table.table-striped tr")}
    crumbs = soup.select("ul.breadcrumb li")
    desc_header = soup.select_one("#product_description")
    desc = desc_header.find_next_sibling("p") if desc_header else None
    rating = soup.select_one("p.star-rating")
    return {
        "title": soup.h1.get_text(strip=True),
        "product_url": url,
        "category": crumbs[2].get_text(strip=True) if len(crumbs) > 2 else None,
        "price_excl_tax": table.get("Price (excl. tax)"),
        "price_incl_tax": table.get("Price (incl. tax)"),
        "tax": table.get("Tax"),
        "rating": rating["class"][1] if rating else None,
        "availability": table.get("Availability"),
        "num_reviews": table.get("Number of reviews"),
        "upc": table.get("UPC"),
        "description": desc.get_text(strip=True) if desc else None,
        "collected_at": datetime.now().isoformat(timespec="seconds"),
    }

### What this cell does:
This cell defines the functions used for scraping and does not download anything yet. One function downloads a page with a short delay, retries on errors and counts requests. Another collects all book links from the listing pages. The third reads one book page and extracts its details with a collected_at timestamp. There is no output because these are only function definitions.

In [19]:
soup = get_soup(START_URL)
print("Books on page 1:", len(soup.select("article.product_pod")))
print("Next-page link:", soup.select_one("li.next a")["href"])

cats = soup.select("div.side_categories ul li ul li a")
print("Categories in sidebar:", len(cats))

first = urljoin(START_URL, soup.select_one("article.product_pod h3 a")["href"])
print("Example product page:", first)
print(parse_product(first))

Books on page 1: 20
Next-page link: page-2.html
Categories in sidebar: 50
Example product page: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
{'title': 'A Light in the Attic', 'product_url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'category': 'Poetry', 'price_excl_tax': '£51.77', 'price_incl_tax': '£51.77', 'tax': '£0.00', 'rating': 'Three', 'availability': 'In stock (22 available)', 'num_reviews': '0', 'upc': 'a897fe39b1053632', 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings

### What this cell does:
This cell tests the scraper on the first page and on one product page. It helps me understand how the site works before the full run. Listing pages have a next link for pagination, the sidebar has the category pages, and each book has its own product page. The output shows the books on page 1, the next page link, the number of categories and one example product.

In [20]:
collection_start = datetime.now()

product_urls, n_pages = get_product_urls()
product_urls = list(dict.fromkeys(product_urls))
print("Listing pages visited:", n_pages, "| unique product URLs:", len(product_urls))

rows = []
for i, url in enumerate(product_urls, 1):
    row = parse_product(url)
    if row:
        rows.append(row)
    if i % 100 == 0:
        print(f"{i}/{len(product_urls)} done | requests so far: {request_count}")

collection_end = datetime.now()

df = pd.DataFrame(rows)
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RAW_PATH, index=False, encoding="utf-8", lineterminator="\n")
print("Saved", len(df), "rows to", RAW_PATH)
print("Failed URLs:", failed_urls)

Listing pages visited: 50 | unique product URLs: 1000
100/1000 done | requests so far: 152
200/1000 done | requests so far: 252
300/1000 done | requests so far: 352
400/1000 done | requests so far: 452
500/1000 done | requests so far: 552
600/1000 done | requests so far: 652
700/1000 done | requests so far: 752
800/1000 done | requests so far: 852
900/1000 done | requests so far: 952
1000/1000 done | requests so far: 1052
Saved 1000 rows to C:\Users\DELL\data\E-commerce\data\raw\books_raw.csv
Failed URLs: []


### What this cell does:
This cell runs the full scrape. It visits all 50 listing pages and then every product page. The data is saved as books_raw.csv with one book per row. The values are saved exactly as scraped, and no rows were copied or invented. The output shows 50 pages visited, 1000 unique links found and 1000 rows saved.

In [21]:
ROLL_NUMBERS = ["25L-3042", "25L-3020"]

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("Team roll numbers     :", ROLL_NUMBERS)
print("Source URL            :", "https://books.toscrape.com/")
print("Collection start      :", collection_start.isoformat(timespec="seconds"))
print("Collection end        :", collection_end.isoformat(timespec="seconds"))
print("Requests made         :", request_count)
print("Listing pages visited :", n_pages)
print("Rows collected        :", len(df))
print("collected_at filled   :", df["collected_at"].notna().all())
print("SHA-256 of raw file   :", sha256_of(RAW_PATH))

Team roll numbers     : ['25L-3042', '25L-3020']
Source URL            : https://books.toscrape.com/
Collection start      : 2026-11-19T07:35:25
Collection end        : 2026-11-19T07:50:41
Requests made         : 1052
Listing pages visited : 50
Rows collected        : 1000
collected_at filled   : True
SHA-256 of raw file   : f6129f4d7f32af7fc7dff8eef0fdf57c6aca2051987da258d5dab4fa2b0db802


### What this cell does:
This cell prints the data collection log and the SHA-256 fingerprint of the raw file. It proves the data was collected by our team, and the submitted file must match this fingerprint. Our roll numbers are 25L-3042 and 25L-3020. Collection ran from 07:35:25 to 07:50:41 on 2026-11-19. We made 1052 requests, which includes 2 extra requests from the test in Cell 3. We collected 1000 rows and every row has a collected_at value.

In [22]:
raw = pd.read_csv(RAW_PATH)
print("Rows:", len(raw))
print("Unique product URLs:", raw["product_url"].nunique(), "| Unique UPCs:", raw["upc"].nunique())
print("\nMissing values per column:\n", raw.isna().sum())
print("\nRatings:", raw["rating"].value_counts().to_dict())
print("\nAvailability formats:",
      raw["availability"].str.replace(r"\d+", "N", regex=True).value_counts().to_dict())

Rows: 1000
Unique product URLs: 1000 | Unique UPCs: 1000

Missing values per column:
 title             0
product_url       0
category          0
price_excl_tax    0
price_incl_tax    0
tax               0
rating            0
availability      0
num_reviews       0
upc               0
description       2
collected_at      0
dtype: int64

Ratings: {'One': 226, 'Three': 203, 'Five': 196, 'Two': 196, 'Four': 179}

Availability formats: {'In stock (N available)': 1000}


### What this cell does:
This cell reads the raw file and checks it for duplicates, missing values and inconsistent formats. There are 1000 rows with 1000 unique URLs and 1000 unique UPCs, so there are no duplicates. Only the description column has missing values, in 2 books, and I left them empty. Ratings are text values from One to Five. Availability always has the same format. Prices, tax and reviews are still raw text and will be cleaned in the next step.

In [23]:
cat_counts = {}
for c in cats:
    name, url = c.get_text(strip=True), urljoin(START_URL, c["href"])
    n = 0
    while url:
        s = get_soup(url)
        if s is None:
            break
        n += len(s.select("article.product_pod"))
        nxt = s.select_one("li.next a")
        url = urljoin(url, nxt["href"]) if nxt else None
    cat_counts[name] = n

check = pd.DataFrame({"category_pages": cat_counts,
                      "scraped": raw["category"].value_counts().to_dict()}).fillna(0).astype(int)
print("All categories match:", (check["category_pages"] == check["scraped"]).all())

All categories match: True


### What this cell does:
This cell visits every category page and counts the books listed. It compares these counts with the categories in my scraped data. The output shows True, so all categories match and no book was missed or counted twice. These extra requests were only for checking and are not part of the 1052 requests in the collection log.

In [13]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)

books = pd.read_csv(RAW_PATH)
display(books.head())
display(books.tail())

,title,product_url,category,price_excl_tax,price_incl_tax,tax,rating,availability,num_reviews,upc,description,collected_at
0,A Light in the Attic,https://books.toscrape.com/catalogue...,Poetry,£51.77,£51.77,£0.00,Three,In stock (22 available),0,a897fe39b1053632,It's hard to imagine a world without...,2026-11-19T07:36:10
1,Tipping the Velvet,https://books.toscrape.com/catalogue...,Historical Fiction,£53.74,£53.74,£0.00,One,In stock (20 available),0,90fa61229261140a,"""Erotic and absorbing...Written with...",2026-11-19T07:36:11
2,Soumission,https://books.toscrape.com/catalogue...,Fiction,£50.10,£50.10,£0.00,One,In stock (20 available),0,6957f44c3847a760,Dans une France assez proche de la n...,2026-11-19T07:36:12
3,Sharp Objects,https://books.toscrape.com/catalogue...,Mystery,£47.82,£47.82,£0.00,Four,In stock (20 available),0,e00eb4fd7b871a48,"WICKED above her hipbone, GIRL acros...",2026-11-19T07:36:13
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue...,History,£54.23,£54.23,£0.00,Five,In stock (20 available),0,4165285e1663650f,From a renowned historian comes a gr...,2026-11-19T07:36:14


,title,product_url,category,price_excl_tax,price_incl_tax,tax,rating,availability,num_reviews,upc,description,collected_at
995,Alice in Wonderland (Alice's Adventu...,https://books.toscrape.com/catalogue...,Classics,£55.53,£55.53,£0.00,One,In stock (1 available),0,cd2a2a70dd5d176d,NaN,2026-11-19T07:50:37
996,"Ajin: Demi-Human, Volume 1 (Ajin: De...",https://books.toscrape.com/catalogue...,Sequential Art,£57.06,£57.06,£0.00,Four,In stock (1 available),0,bfd5e1701c862ac3,High school student Kei Nagai is str...,2026-11-19T07:50:38
997,A Spy's Devotion (The Regency Spies ...,https://books.toscrape.com/catalogue...,Historical Fiction,£16.97,£16.97,£0.00,Five,In stock (1 available),0,19fec36a1dfb4c16,"In England’s Regency era, manners an...",2026-11-19T07:50:39
998,1st to Die (Women's Murder Club #1),https://books.toscrape.com/catalogue...,Mystery,£53.98,£53.98,£0.00,One,In stock (1 available),0,f684a82adc49f011,"James Patterson, bestselling author ...",2026-11-19T07:50:40
999,"1,000 Places to See Before You Die",https://books.toscrape.com/catalogue...,Travel,£26.08,£26.08,£0.00,Five,In stock (1 available),0,228ba5e7577e1d49,"Around the World, continent by conti...",2026-11-19T07:50:41


### What this cell does:
This cell loads the raw file and shows the first and last five records. Each row is one book, and the values are still raw text.

In [14]:
print("Rows:", books.shape[0])
print("Columns:", books.shape[1])
print("Column names:", list(books.columns))

Rows: 1000
Columns: 12
Column names: ['title', 'product_url', 'category', 'price_excl_tax', 'price_incl_tax', 'tax', 'rating', 'availability', 'num_reviews', 'upc', 'description', 'collected_at']


### What this cell shows:
The dataset has 1000 rows and 12 columns. Each row is one book.

In [15]:
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   title           1000 non-null   str  
 1   product_url     1000 non-null   str  
 2   category        1000 non-null   str  
 3   price_excl_tax  1000 non-null   str  
 4   price_incl_tax  1000 non-null   str  
 5   tax             1000 non-null   str  
 6   rating          1000 non-null   str  
 7   availability    1000 non-null   str  
 8   num_reviews     1000 non-null   int64
 9   upc             1000 non-null   str  
 10  description     998 non-null    str  
 11  collected_at    1000 non-null   str  
dtypes: int64(1), str(11)
memory usage: 93.9 KB


### what this cell shows:
All columns have 1000 non-null values except description, which has 998. Only num_reviews is stored as a number, and the other 11 columns are stored as text.

In [16]:
numeric_cols = books.select_dtypes(include="number").columns.tolist()
boolean_cols = books.select_dtypes(include="bool").columns.tolist()
other_cols = books.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print("Stored as numbers:", numeric_cols)
print("Stored as boolean:", boolean_cols)
print("Stored as text:", other_cols)

Stored as numbers: ['num_reviews']
Stored as boolean: []
Stored as text: ['title', 'product_url', 'category', 'price_excl_tax', 'price_incl_tax', 'tax', 'rating', 'availability', 'upc', 'description', 'collected_at']


### What this cell shows:
Only num_reviews is stored as a number, and there are no boolean columns. By meaning, the price and tax columns are numerical but stored as text. Category, rating and availability are categorical, title and description are text, and product_url and upc are identifiers.

In [17]:
display(books.describe())
display(books.describe(include="all").T)

,num_reviews
count,1000.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
title,1000,999,The Star-Touched Queen,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_url,1000,1000,https://books.toscrape.com/catalogue...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,1000,50,Default,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_excl_tax,1000,903,£44.18,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_incl_tax,1000,903,£44.18,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tax,1000,1,£0.00,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,1000,5,One,226,NaN,NaN,NaN,NaN,NaN,NaN,NaN
availability,1000,21,In stock (3 available),196,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_reviews,1000.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
upc,1000,1000,a897fe39b1053632,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### What this cell shows:
Only num_reviews is numeric, and it has just one unique value, so its statistics are the same for every book. The second table summarizes all columns, including the most frequent value of each text column.

In [18]:
print(books.nunique(), "\n")
for col in ["category", "rating", "availability"]:
    print(col, "- unique values:", books[col].nunique())
    print(books[col].value_counts().head(15), "\n")
print(books["rating"].value_counts().reindex(["One", "Two", "Three", "Four", "Five"]))

title              999
product_url       1000
category            50
price_excl_tax     903
price_incl_tax     903
tax                  1
rating               5
availability        21
num_reviews          1
upc               1000
description        998
collected_at       855
dtype: int64 

category - unique values: 50
category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               19
History                18
Name: count, dtype: int64 

rating - unique values: 5
rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64 

availability - unique values: 21
availability
In stock (3 available)     196
In stock (14 available)    136
In stock (1 availab

### What this cell shows:
There are 50 categories, and the largest is Default with 152 books. Default and Add a comment are labels used by the website and are not real genres. Rating has five values from One to Five. Availability has 21 values only because the stock number changes. Tax and num_reviews each have a single unique value.

In [22]:
print(books["tax"].unique(), books["num_reviews"].unique())

<StringArray>
['£0.00']
Length: 1, dtype: str [0]


### What this cell shows:
This cell shows the unique values of tax and num_reviews. Tax is always ['£0.00'], and num_reviews is always 0.

In [19]:
print(books.memory_usage(deep=True))
total = books.memory_usage(deep=True).sum()
print("Total memory:", round(total / 1024, 1), "KB")

Index                 132
title               89526
product_url        138115
category            58516
price_excl_tax      79000
price_incl_tax      79000
tax                 77000
rating              52781
availability        71420
num_reviews          8000
upc                 65000
description       2342675
collected_at        68000
dtype: int64
Total memory: 3055.8 KB


### What this cell does:
The dataset uses about 3056 KB of memory, and most of it comes from the description column because it holds long text.

In [20]:
missing = books.isna().sum()
percent = (missing / len(books) * 100).round(2)
print(pd.DataFrame({"missing": missing, "percent": percent}))

                missing  percent
title                 0      0.0
product_url           0      0.0
category              0      0.0
price_excl_tax        0      0.0
price_incl_tax        0      0.0
tax                   0      0.0
rating                0      0.0
availability          0      0.0
num_reviews           0      0.0
upc                   0      0.0
description           2      0.2
collected_at          0      0.0


### What this cell shows:
Only description has missing values, 2 out of 1000 books, which is 0.2 percent.

In [21]:
print("Fully duplicated rows:", books.duplicated().sum())
print("Duplicate product URLs:", books.duplicated(subset="product_url").sum())
print("Duplicate UPCs:", books.duplicated(subset="upc").sum())
print("Repeated titles:", books["title"].duplicated().sum())

Fully duplicated rows: 0
Duplicate product URLs: 0
Duplicate UPCs: 0
Repeated titles: 1


### What this cell shows:
There are no duplicate rows, URLs or UPCs. One title appears twice, but the two books have different URLs and UPCs.

In [23]:
books[books["title"].duplicated(keep=False)]

,title,product_url,category,price_excl_tax,price_incl_tax,tax,rating,availability,num_reviews,upc,description,collected_at
236,The Star-Touched Queen,https://books.toscrape.com/catalogue...,Fantasy,£46.02,£46.02,£0.00,Five,In stock (14 available),0,1528279aec1f3dce,Fate and fortune. Power and passion....,2026-11-19T07:39:31
358,The Star-Touched Queen,https://books.toscrape.com/catalogue...,Fantasy,£32.30,£32.30,£0.00,Five,In stock (12 available),0,4a7a25be293ad678,Fate and fortune. Power and passion....,2026-11-19T07:41:13


### What this cell shows:
This cell shows the two books that share a title. Their URLs and UPCs are different, so they are different books.

In [25]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

TARGET_REG = "price"
TARGET_CLF = "rating_class"

orig_missing = books.isna().sum()
report = pd.DataFrame({"missing": orig_missing,
                       "percent": (orig_missing / len(books) * 100).round(2)})
print(report)

                missing  percent
title                 0      0.0
product_url           0      0.0
category              0      0.0
price_excl_tax        0      0.0
price_incl_tax        0      0.0
tax                   0      0.0
rating                0      0.0
availability          0      0.0
num_reviews           0      0.0
upc                   0      0.0
description           2      0.2
collected_at          0      0.0


### What this cell shows:
Only description has missing values in the raw data, 2 out of 1000 books or 0.2 percent. The targets are price for regression and rating_class for classification.

In [26]:
clean = books.copy()
clean["price"] = clean["price_excl_tax"].str.replace(r"[^\d.]", "", regex=True).astype(float)
price_incl = clean["price_incl_tax"].str.replace(r"[^\d.]", "", regex=True).astype(float)
print("Price incl. tax equals price excl. tax:", (clean["price"] == price_incl).all())
clean["stock"] = clean["availability"].str.extract(r"\((\d+) available\)", expand=False).astype(float)
clean["rating_num"] = clean["rating"].map({"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5})
clean["description"] = clean["description"].fillna("")
clean["desc_length"] = clean["description"].str.len().astype(float)
clean["title_length"] = clean["title"].str.len().astype(float)
clean["collected_at"] = pd.to_datetime(clean["collected_at"])
clean["rating_class"] = pd.cut(clean["rating_num"], bins=[0, 2, 3, 5],labels=["Low", "Medium", "High"])
display(clean[["price", "stock", "rating", "rating_num", "rating_class", "desc_length", "title_length"]].head())

Price incl. tax equals price excl. tax: True


,price,stock,rating,rating_num,rating_class,desc_length,title_length
0,51.77,22.0,Three,3,Medium,1017.0,20.0
1,53.74,20.0,One,1,Low,1029.0,18.0
2,50.10,20.0,One,1,Low,1093.0,10.0
3,47.82,20.0,Four,4,High,1635.0,13.0
4,54.23,20.0,Five,5,High,1945.0,37.0


### What this cell does:
This cell makes a working copy and converts the text into usable columns such as price, stock and rating number. It also adds description and title length. Price including tax equals price excluding tax because the tax is always zero. Ratings are grouped into Low (1 and 2 stars), Medium (3 stars) and High (4 and 5 stars).

In [27]:
print(clean["rating_class"].value_counts().reindex(["Low", "Medium", "High"]))
print((clean["rating_class"].value_counts(normalize=True) * 100).round(1))

constant_cols = [c for c in clean.columns if clean[c].nunique() == 1]
print("Constant columns, not used as inputs:", constant_cols)

rating_class
Low       422
Medium    203
High      375
Name: count, dtype: int64
rating_class
Low       42.2
High      37.5
Medium    20.3
Name: proportion, dtype: float64
Constant columns, not used as inputs: ['tax', 'num_reviews']


### What this cell shows:
The classes are Low 422, Medium 203 and High 375, so Medium is the smallest class. Tax and num_reviews are constant, so they are not used as inputs.

In [28]:
cols_no_time = [c for c in clean.columns if c != "collected_at"]
print("Exact duplicate rows:", clean.duplicated().sum())
print("Duplicates ignoring collected_at:", clean.duplicated(subset=cols_no_time).sum())
before = len(clean)
clean = clean.drop_duplicates(subset=cols_no_time).reset_index(drop=True)
print("Rows before:", before, "| removed:", before - len(clean), "| rows after:", len(clean))

Exact duplicate rows: 0
Duplicates ignoring collected_at: 0
Rows before: 1000 | removed: 0 | rows after: 1000


### What this cell shows:
No duplicate records were found, so 0 rows were removed and 1000 rows remain.

In [29]:
MISSING_COLS = ["stock", "desc_length", "category"]
assert TARGET_REG not in MISSING_COLS and TARGET_CLF not in MISSING_COLS

rng = np.random.default_rng(SEED)
demo = clean.copy()
injected = {}
for col in MISSING_COLS:
    idx = rng.choice(demo.index.to_numpy(), size=int(0.05 * len(demo)), replace=False)
    injected[col] = idx
    demo.loc[idx, col] = np.nan

miss = demo[MISSING_COLS].isna().sum()
print(pd.DataFrame({"missing": miss, "percent": (miss / len(demo) * 100).round(2)}))

             missing  percent
stock             50      5.0
desc_length       50      5.0
category          50      5.0


### What this cell does:
This cell adds about 5 percent missing values to stock, description length and category on a copy of the data. The targets were not changed.

In [30]:
imputed = demo.copy()
for col in ["stock", "desc_length"]:
    skew = demo[col].skew()
    use_median = abs(skew) > 1
    value = demo[col].median() if use_median else demo[col].mean()
    imputed[col] = demo[col].fillna(value)
    print(col, "| skew:", round(skew, 2), "| used:", "median" if use_median else "mean", "| value:", round(value, 2))

imputed["category"] = demo["category"].fillna(demo["category"].mode()[0])

print(pd.DataFrame({"before": demo[MISSING_COLS].isna().sum(),
                    "after": imputed[MISSING_COLS].isna().sum()}))

stock | skew: 0.24 | used: mean | value: 8.46
desc_length | skew: 2.72 | used: median | value: 1352.5
             before  after
stock            50      0
desc_length      50      0
category         50      0


### What this cell does:
Stock is not skewed, so I filled it with the mean. Description length is skewed, so I used the median. Category was filled with the most frequent value. Every column went from 50 missing values to 0.

In [31]:
for col in ["stock", "desc_length"]:
    idx = injected[col]
    err = (imputed.loc[idx, col] - clean.loc[idx, col]).abs().mean()
    print(col, "mean absolute error:", round(err, 2))

idx = injected["category"]
share = (imputed.loc[idx, "category"] == clean.loc[idx, "category"]).mean()
print("category: share of imputed values that were correct:", round(share, 3))

stock mean absolute error: 6.19
desc_length mean absolute error: 338.1
category: share of imputed values that were correct: 0.06


### What this cell shows:
The average error is 6.19 for stock and 338.1 for description length, and only 6 percent of the categories were correct. Filling with a typical value cannot recover the true value.

In [32]:
onehot = pd.get_dummies(clean["category"], prefix="cat", dtype=int)
print("One-hot shape:", onehot.shape)
display(onehot.head())

print(clean[["rating", "rating_num", "rating_class"]].drop_duplicates().sort_values("rating_num"))

One-hot shape: (1000, 50)


,cat_Academic,cat_Add a comment,cat_Adult Fiction,cat_Art,cat_Autobiography,cat_Biography,cat_Business,cat_Childrens,cat_Christian,cat_Christian Fiction,cat_Classics,cat_Contemporary,cat_Crime,cat_Cultural,cat_Default,cat_Erotica,cat_Fantasy,cat_Fiction,cat_Food and Drink,cat_Health,cat_Historical,cat_Historical Fiction,cat_History,cat_Horror,cat_Humor,cat_Music,cat_Mystery,cat_New Adult,cat_Nonfiction,cat_Novels,cat_Paranormal,cat_Parenting,cat_Philosophy,cat_Poetry,cat_Politics,cat_Psychology,cat_Religion,cat_Romance,cat_Science,cat_Science Fiction,cat_Self Help,cat_Sequential Art,cat_Short Stories,cat_Spirituality,cat_Sports and Games,cat_Suspense,cat_Thriller,cat_Travel,cat_Womens Fiction,cat_Young Adult
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


   rating  rating_num rating_class
1     One           1          Low
10    Two           2          Low
0   Three           3       Medium
3    Four           4         High
4    Five           5         High


### What this cell does:
Category has no order, so I used one-hot encoding. The star rating has an order, so I kept it as numbers from 1 to 5.

In [33]:
numeric_features = ["stock", "desc_length", "title_length"]

minmax = pd.DataFrame(MinMaxScaler().fit_transform(clean[numeric_features]), columns=numeric_features)
zscore = pd.DataFrame(StandardScaler().fit_transform(clean[numeric_features]), columns=numeric_features)

print("Min-Max (min and max):\n", minmax.agg(["min", "max"]).round(3))
print("Z-score (mean and std):\n", pd.DataFrame({"mean": zscore.mean(), "std": zscore.std(ddof=0)}).round(3))

Min-Max (min and max):
      stock  desc_length  title_length
min    0.0          0.0           0.0
max    1.0          1.0           1.0
Z-score (mean and std):
               mean  std
stock         -0.0  1.0
desc_length    0.0  1.0
title_length  -0.0  1.0


### What this cell shows:
Min-Max normalization gives every column a range of 0 to 1. Z-score standardization gives every column a mean of 0 and a standard deviation of 1. Scaling puts the columns on the same range.

In [34]:
reg_features_num = ["stock", "desc_length", "title_length", "rating_num"]
reg_features_cat = ["category"]
clf_features_num = ["price", "stock", "desc_length", "title_length"]
clf_features_cat = ["category"]
assert not {"price", "price_excl_tax", "price_incl_tax", "tax"} & set(reg_features_num + reg_features_cat)
assert not {"rating", "rating_num", "rating_class"} & set(clf_features_num + clf_features_cat)
assert not set(constant_cols) & set(reg_features_num + reg_features_cat + clf_features_num + clf_features_cat)
print("Regression inputs:", reg_features_num + reg_features_cat)
print("Classification inputs:", clf_features_num + clf_features_cat)
print("No leakage between targets and inputs")

Regression inputs: ['stock', 'desc_length', 'title_length', 'rating_num', 'category']
Classification inputs: ['price', 'stock', 'desc_length', 'title_length', 'category']
No leakage between targets and inputs


In [35]:
PROCESSED_PATH = ROOT / "data" / "processed" / "books_clean.csv"
clean.to_csv(PROCESSED_PATH, index=False)
print("Saved", len(clean), "rows to", PROCESSED_PATH)

Saved 1000 rows to C:\Users\DELL\data\E-commerce\data\processed\books_clean.csv
